# PHARMALENS AI — PROJECT 11
## External Data Intelligence

Purpose:
- Retrieve trusted external data through APIs.
- Normalize external sources into a common PharmaLens structure.
- Build epidemiology / patient-flow inputs.
- Create benchmarking inputs for forecasting and strategic analysis.
- Keep source, retrieval time, and evidence metadata for traceability.

This notebook is designed to work independently of Project 11A Corporate Data Integration.


In [1]:
# CELL 01 — Project initialization

import sys
import os
import json
import time
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import requests

PROJECT_ROOT = Path.cwd().resolve().parents[0]

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
EXTERNAL_RAW_DIR = DATA_DIR / "external" / "raw"
EXTERNAL_PROCESSED_DIR = DATA_DIR / "external" / "processed"
EXTERNAL_RAW_DIR.mkdir(parents=True, exist_ok=True)
EXTERNAL_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("PharmaLens AI — Project 11")
print("PROJECT_ROOT:", PROJECT_ROOT)


PharmaLens AI — Project 11
PROJECT_ROOT: D:\learning\Epsilon\Data Science\PDF lec\Decodelab\PharmaLens AI


In [2]:
# CELL 02 — Generic GET request with retry

def api_get(url, headers=None, params=None, timeout=60, retries=3):
    headers = headers or {}
    last_error = None

    for attempt in range(retries):
        try:
            response = requests.get(
                url,
                headers=headers,
                params=params,
                timeout=timeout
            )
            response.raise_for_status()
            return response
        except requests.RequestException as exc:
            last_error = exc
            if attempt < retries - 1:
                time.sleep(2 ** attempt)

    raise last_error


In [3]:
# CELL 03 — JSON helper

def fetch_json(url, headers=None, params=None, timeout=60):
    response = api_get(
        url=url,
        headers=headers,
        params=params,
        timeout=timeout
    )
    return response.json()

def json_to_dataframe(payload, records_key=None):
    if payload is None:
        return pd.DataFrame()

    if records_key and isinstance(payload, dict):
        payload = payload.get(records_key, [])

    if isinstance(payload, dict):
        for key in ["records", "data", "results", "items"]:
            if key in payload:
                payload = payload[key]
                break

    if isinstance(payload, list):
        return pd.json_normalize(payload)

    if isinstance(payload, dict):
        return pd.json_normalize([payload])

    return pd.DataFrame()


In [4]:
# CELL 04 — Column normalization

def normalize_column_name(column):
    column = str(column).strip()
    column = (
        column.replace(".", "_")
              .replace("-", "_")
              .replace("/", "_")
              .replace(" ", "_")
    )
    while "__" in column:
        column = column.replace("__", "_")
    return column.strip("_")

def normalize_dataframe_columns(df):
    df = df.copy()
    df.columns = [normalize_column_name(c) for c in df.columns]
    return df


In [5]:
# CELL 05 — Source registry

EXTERNAL_SOURCE_REGISTRY = pd.DataFrame([
    {
        "Source": "WHO",
        "Category": "Epidemiology / Health",
        "Access": "API / Public data",
        "Trust_Level": "High",
        "Use": "Population and health indicators"
    },
    {
        "Source": "World Bank",
        "Category": "Economic / Population",
        "Access": "API",
        "Trust_Level": "High",
        "Use": "Population and macroeconomic indicators"
    },
    {
        "Source": "PubMed",
        "Category": "Clinical Evidence",
        "Access": "API",
        "Trust_Level": "High",
        "Use": "Scientific literature and evidence"
    },
    {
        "Source": "Regulatory Authority",
        "Category": "Regulatory",
        "Access": "API / Dataset / Manual",
        "Trust_Level": "High",
        "Use": "Approvals, safety and regulatory signals"
    },
    {
        "Source": "Disease / Epidemiology Dataset",
        "Category": "Epidemiology",
        "Access": "API / Dataset",
        "Trust_Level": "High",
        "Use": "Disease burden and patient-flow inputs"
    }
])

display(EXTERNAL_SOURCE_REGISTRY)


,Source,Category,Access,Trust_Level,Use
0,WHO,Epidemiology / Health,API / Public data,High,Population and health indicators
1,World Bank,Economic / Population,API,High,Population and macroeconomic indicators
2,PubMed,Clinical Evidence,API,High,Scientific literature and evidence
3,Regulatory Authority,Regulatory,API / Dataset / Manual,High,"Approvals, safety and regulatory signals"
4,Disease / Epidemiology Dataset,Epidemiology,API / Dataset,High,Disease burden and patient-flow inputs


In [6]:
# CELL 06 — World Bank API loader

WORLD_BANK_URL = "https://api.worldbank.org/v2/country/{country}/indicator/{indicator}"

def world_bank_indicator(country="EGY", indicator="SP.POP.TOTL", per_page=1000):
    url = WORLD_BANK_URL.format(
        country=country,
        indicator=indicator
    )

    payload = fetch_json(
        url,
        params={
            "format": "json",
            "per_page": per_page
        }
    )

    if not isinstance(payload, list) or len(payload) < 2:
        return pd.DataFrame()

    records = payload[1]
    df = pd.json_normalize(records)
    df = normalize_dataframe_columns(df)

    return df

# Example:
# egypt_population = world_bank_indicator("EGY", "SP.POP.TOTL")
# display(egypt_population.head())


In [7]:
# CELL 07 — WHO API framework

# WHO API endpoints and availability can change.
# Keep the endpoint configurable rather than hard-coding a fragile dataset.

WHO_API_BASE_URL = os.getenv(
    "PHARMALENS_WHO_API_BASE_URL",
    ""
)

def fetch_who_data(endpoint, params=None):
    if not WHO_API_BASE_URL:
        raise RuntimeError(
            "Set PHARMALENS_WHO_API_BASE_URL before using the WHO connector."
        )

    url = WHO_API_BASE_URL.rstrip("/") + "/" + endpoint.lstrip("/")

    payload = fetch_json(
        url,
        params=params
    )

    return json_to_dataframe(payload)

print("WHO connector framework ready.")


WHO connector framework ready.


In [8]:
# CELL 08 — PubMed E-utilities search

PUBMED_ESEARCH_URL = (
    "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
)

def pubmed_search(term, retmax=20):
    response = api_get(
        PUBMED_ESEARCH_URL,
        params={
            "db": "pubmed",
            "term": term,
            "retmode": "json",
            "retmax": retmax
        }
    )

    payload = response.json()

    ids = (
        payload.get("esearchresult", {})
               .get("idlist", [])
    )

    return pd.DataFrame({
        "PubMed_ID": ids,
        "Search_Term": term
    })

# Example:
# pubmed_results = pubmed_search("diabetes Egypt", retmax=20)
# display(pubmed_results.head())


In [9]:
# CELL 09 — External data metadata

def add_source_metadata(
    df,
    source,
    category,
    retrieval_timestamp=None
):
    df = df.copy()

    if retrieval_timestamp is None:
        retrieval_timestamp = datetime.now(timezone.utc).isoformat()

    df["Data_Source"] = source
    df["Source_Category"] = category
    df["Retrieved_At_UTC"] = retrieval_timestamp

    return df


In [10]:
# CELL 10 — Patient-flow model

def calculate_patient_flow(
    population,
    prevalence_rate,
    diagnosis_rate,
    treatment_rate,
    eligible_rate=1.0,
    access_rate=1.0
):
    population = float(population)
    prevalence_rate = float(prevalence_rate)
    diagnosis_rate = float(diagnosis_rate)
    treatment_rate = float(treatment_rate)
    eligible_rate = float(eligible_rate)
    access_rate = float(access_rate)

    disease_population = population * prevalence_rate
    diagnosed_patients = disease_population * diagnosis_rate
    treatment_population = diagnosed_patients * treatment_rate
    eligible_patients = treatment_population * eligible_rate
    accessible_patients = eligible_patients * access_rate

    return {
        "Population": population,
        "Disease_Population": disease_population,
        "Diagnosed_Patients": diagnosed_patients,
        "Treated_Patients": treatment_population,
        "Eligible_Patients": eligible_patients,
        "Accessible_Patients": accessible_patients
    }


In [11]:
# CELL 11 — Patient-flow table builder

def build_patient_flow_table(
    rows
):
    result = []

    for row in rows:
        metrics = calculate_patient_flow(
            population=row["Population"],
            prevalence_rate=row["Prevalence_Rate"],
            diagnosis_rate=row["Diagnosis_Rate"],
            treatment_rate=row["Treatment_Rate"],
            eligible_rate=row.get("Eligible_Rate", 1.0),
            access_rate=row.get("Access_Rate", 1.0)
        )

        result.append({
            **row,
            **metrics
        })

    return pd.DataFrame(result)


In [12]:
# CELL 12 — Patient-flow demonstration

patient_flow_demo = build_patient_flow_table([
    {
        "Country": "Egypt",
        "Disease": "Example Disease",
        "Year": 2026,
        "Population": 110_000_000,
        "Prevalence_Rate": 0.05,
        "Diagnosis_Rate": 0.60,
        "Treatment_Rate": 0.70,
        "Eligible_Rate": 0.80,
        "Access_Rate": 0.85
    }
])

display(patient_flow_demo)


,Country,Disease,Year,Population,Prevalence_Rate,Diagnosis_Rate,Treatment_Rate,Eligible_Rate,Access_Rate,Disease_Population,Diagnosed_Patients,Treated_Patients,Eligible_Patients,Accessible_Patients
0,Egypt,Example Disease,2026,110000000.0,0.05,0.6,0.7,0.8,0.85,5500000.0,3300000.0,2310000.0,1848000.0,1570800.0


In [13]:
# CELL 13 — Benchmarking framework

def calculate_market_benchmark(
    market_df,
    value_column="Sales_Value",
    group_column="Brand"
):
    df = market_df.copy()

    summary = (
        df.groupby(group_column)[value_column]
          .sum()
          .reset_index()
          .rename(columns={value_column: "Benchmark_Value"})
    )

    total = summary["Benchmark_Value"].sum()

    summary["Benchmark_Share_%"] = np.where(
        total > 0,
        summary["Benchmark_Value"] / total * 100,
        np.nan
    )

    return summary.sort_values(
        "Benchmark_Value",
        ascending=False
    )

print("Benchmarking framework ready.")


Benchmarking framework ready.


In [14]:
# CELL 14 — Growth benchmark

def calculate_growth_benchmark(
    df,
    value_column="Sales_Value",
    time_column="Year",
    group_column="Brand"
):
    work = df.copy()

    work = work.sort_values(
        [group_column, time_column]
    )

    first = (
        work.groupby(group_column)[value_column]
            .first()
    )

    last = (
        work.groupby(group_column)[value_column]
            .last()
    )

    periods = (
        work.groupby(group_column)[time_column]
            .nunique()
            .clip(lower=2)
    )

    benchmark = pd.DataFrame({
        "First_Value": first,
        "Last_Value": last,
        "Periods": periods
    }).reset_index()

    benchmark["CAGR_%"] = np.where(
        (benchmark["First_Value"] > 0)
        & (benchmark["Periods"] > 1),
        (
            (benchmark["Last_Value"] /
             benchmark["First_Value"])
            ** (1 / (benchmark["Periods"] - 1))
            - 1
        ) * 100,
        np.nan
    )

    return benchmark


In [15]:
# CELL 15 — Save external datasets

def save_external_data(df, dataset_name, processed=True):
    directory = (
        EXTERNAL_PROCESSED_DIR
        if processed
        else EXTERNAL_RAW_DIR
    )

    path = directory / f"{dataset_name}.parquet"
    df.to_parquet(path, index=False)

    return path

# Example:
# save_external_data(patient_flow_demo, "patient_flow")


In [16]:
# CELL 16 — Evidence quality scoring

def evidence_quality_score(
    source_type,
    peer_reviewed=False,
    official_authority=False,
    recent=True
):
    score = 0

    if official_authority:
        score += 50

    if peer_reviewed:
        score += 30

    if recent:
        score += 20

    if score >= 80:
        label = "High"
    elif score >= 50:
        label = "Moderate"
    else:
        label = "Low"

    return score, label


In [17]:
# CELL 17 — Final status

print("=" * 70)
print("PHARMALENS AI — PROJECT 11")
print("EXTERNAL DATA INTELLIGENCE")
print("=" * 70)
print("Generic API framework       : READY")
print("World Bank connector        : READY")
print("WHO connector framework     : READY")
print("PubMed connector            : READY")
print("Patient-flow model          : READY")
print("Benchmarking framework      : READY")
print("Evidence metadata           : READY")
print("External parquet storage    : READY")
print("=" * 70)


PHARMALENS AI — PROJECT 11
EXTERNAL DATA INTELLIGENCE
Generic API framework       : READY
World Bank connector        : READY
WHO connector framework     : READY
PubMed connector            : READY
Patient-flow model          : READY
Benchmarking framework      : READY
Evidence metadata           : READY
External parquet storage    : READY
